In [1]:
import os
os.chdir('../')

In [2]:
import numpy as np
import torch
import torch.nn.functional as F

pt_file = 'experiments/imagenet64/unipc_order3_200/image_samples/images/target.pt'
data = torch.load(pt_file)
noise = data['noise_raw']
sample = data['sample_raw']
print(noise.shape, sample.shape)

torch.Size([128, 3, 64, 64]) torch.Size([128, 3, 64, 64])


In [3]:
import sys
import torch
import numpy as np
import random

from main import parse_args_and_config, Diffusion

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.benchmark = False

2025-05-18 13:42:15.921293: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-18 13:42:15.939642: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-18 13:42:15.939668: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-18 13:42:15.940307: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-18 13:42:15.943647: I tensorflow/core/platform/cpu_feature_guar

In [6]:
sys.argv = [
            "main.py",
            "--config", "imagenet64.yml",
            "--sample",
            "--dpm_solver_type", "data_prediction",
            "--dpm_solver_order", "3",
            "--skip_type", "logSNR",
            "--ni",
            "--sample_type", "rbf_solver",
            "--timesteps", "5",
            "--shape_dir", "shape_dir",
            "--lower_order_final",
        ]

args, config = parse_args_and_config()
diffusion = Diffusion(args, config, rank=0)
diffusion.prepare_model()
diffusion.model.eval()
device = diffusion.device

for order in [3, 4]:
    for NFE in [5, 6, 8, 10, 12, 15, 20, 25, 30, 35, 40]:
        diffusion.args.dpm_solver_order = order
        diffusion.args.timesteps = NFE
        
        noise_batch = noise.to(device)
        sample_batch = sample.to(device)
        pred, _ = diffusion.sample_image(noise_batch, diffusion.model, target=sample_batch)
        loss = F.mse_loss(pred, sample_batch)
        print('NFE :', NFE, 'order :', order, 'loss :', loss)


INFO - main.py - 2025-05-18 13:47:32,999 - Using device: cuda
INFO - main.py - 2025-05-18 13:47:32,999 - Using device: cuda


[prepare_model] Model is ready.
def sample_with_optim start!!!
noise_target.shape: torch.Size([128, 3, 64, 64]), data_target.shape: torch.Size([128, 3, 64, 64]), steps: 5, order: 3, skip_type: logSNR, lower_order_final: True
shape_dir/NFE=5,p=3.npz shape file saved!
NFE : 5 order : 3 loss : tensor(0.0420, device='cuda:0')
def sample_with_optim start!!!
noise_target.shape: torch.Size([128, 3, 64, 64]), data_target.shape: torch.Size([128, 3, 64, 64]), steps: 6, order: 3, skip_type: logSNR, lower_order_final: True
shape_dir/NFE=6,p=3.npz shape file saved!
NFE : 6 order : 3 loss : tensor(0.0276, device='cuda:0')
def sample_with_optim start!!!
noise_target.shape: torch.Size([128, 3, 64, 64]), data_target.shape: torch.Size([128, 3, 64, 64]), steps: 8, order: 3, skip_type: logSNR, lower_order_final: True
shape_dir/NFE=8,p=3.npz shape file saved!
NFE : 8 order : 3 loss : tensor(0.0129, device='cuda:0')
def sample_with_optim start!!!
noise_target.shape: torch.Size([128, 3, 64, 64]), data_target

In [7]:
print('done')

done
